In [45]:
from langsmith import Client
from qdrant_client import QdrantClient
import instructor
from openai import OpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
import os
import openai
from dotenv import load_dotenv
load_dotenv()
from qdrant_client.models import Distance,VectorParams,PointStruct,Filter,FieldCondition,MatchValue,Prefetch,Document,models

In [30]:
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
collection_name = os.getenv("QDRANT_COLLECTION_NAME")


In [31]:
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY,timeout=60)

### Download an example refernce data point from Langsmith

In [6]:
langsmith_client=Client(api_key=os.environ['LANGCHAIN_API_KEY'])

datasets=langsmith_client.read_dataset(
    dataset_name="shopmart_eval_dataset"
)

In [7]:
datasets

Dataset(name='shopmart_eval_dataset', description='Datasets for evaluating Rag pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('fbca5ce2-cdbf-44b6-859c-27caa2777a1f'), created_at=datetime.datetime(2026, 11, 10, 14, 53, 57, 832561, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 11, 10, 14, 53, 57, 832561, tzinfo=TzInfo(0)), example_count=80, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'Windows-11-10.0.26300-SP0', 'sdk_version': '0.10.1', 'runtime_version': '3.12.14', 'langchain_version': '1.3.12', 'py_implementation': 'CPython', 'langchain_core_version': '1.4.9'}})

In [11]:
output_data=list(langsmith_client.list_examples(dataset_id=datasets.id,limit=100))

In [52]:
reference_output=output_data[2].outputs
reference_output

{'parent_asin': ['B0B9SY7F4Q'],
 'ground_truth': 'Yes — the ZAMKOL Bluetooth speaker (ZK706 Pro) includes an 8000mAh battery that can act as an emergency charger (power bank), supports up to 24 hours playtime and true wireless stereo pairing (TWS) to pair two speakers for 100W stereo output.',
 'reference_context_ids': ['B0B9SY7F4Q'],
 'reference_description': ['ZAMKOL Bluetooth Speaker, 50W Loud Portable Speaker with Deep Bass, IPX7 Waterproof, 8000mAh Emergency Charger, 24H Playtime, RGB Light, Outdoor Wireless Speakers for Party, Travel🎵 【HD Sound Quality & Powerful Bass】 - The wireless bluetooth speaker integrated a purpose-built 81mm NdFeB driver and an extra subwoofer, coupled with unique BAMBU algorithm, which offering the super powerful bass and crystal clear treble. The speaker delivers incredible volume through a dual-precision stereo driver. No distortion even at maximum volume.🎵 【RGB Lighting Effect & Bluetooth 5.0】 - The ZK706 Pro speaker offers 6 different lighting effect

In [54]:
reference_input=output_data[2].inputs
reference_input

{'question': 'Is there a wireless speaker that doubles as a power bank and supports TWS pairing?'}

In [55]:
from rag.reterivalgeneration import rag_pipeline

output = rag_pipeline(question="Is there a wireless speaker that doubles as a power bank and supports TWS pairing?")

In [56]:
output

{'original_output': RAGGenerationResponse(answer='Yes, there are wireless speakers that double as power banks and support True Wireless Stereo (TWS) pairing. For example, the ZAMKOL Bluetooth Speaker (ID: B09YYCBYTR and B0B9SY7F4Q) features a 50W loud output, IPX7 waterproof rating, 8000mAh battery that can be used as an emergency charger (power bank), 24 hours playtime, and supports 100W True Wireless Stereo Pairing (TWS) for immersive stereo sound.', references=[RAGUsedContext(id='B09YYCBYTR', description='ZAMKOL Bluetooth Speaker, 50W Loud Portable Speaker with Deep Bass, IPX7 Waterproof, 8000mAh Emergency Charger, 24H Playtime, RGB Light, Outdoor Wireless Speakers for Party, Travel'), RAGUsedContext(id='B0B9SY7F4Q', description='ZAMKOL Bluetooth Speaker, 50W Loud Portable Speaker with Deep Bass, IPX7 Waterproof, 8000mAh Emergency Charger, 24H Playtime, RGB Light, Outdoor Wireless Speakers for Party, Travel')]),
 'answer': 'Yes, there are wireless speakers that double as power banks

In [57]:
from ragas.dataset_schema import SingleTurnSample 
from ragas.metrics import (
    ContextPrecision,  # Formerly IDBasedContextPrecision
    ContextRecall,     # Formerly IDBasedContextRecall
    Faithfulness,      # Fixed typo: 'FaithFullness' -> 'Faithfulness'
    ResponseRelevancy  # Fixed typo: 'ResponceRelevancy' -> 'ResponseRelevancy'
)

C:\Users\Anurag\AppData\Local\Temp\ipykernel_33096\740285422.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\Anurag\AppData\Local\Temp\ipykernel_33096\740285422.py:2: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import (
C:\Users\Anurag\AppData\Local\Temp\ipykernel_33096\740285422.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\Anurag\AppData\Local\Temp\ipykernel_33096\740

In [58]:
import os

os.environ.pop("LANGCHAIN_TRACING_V2", None)
os.environ.pop("LANGCHAIN_API_KEY", None)
os.environ["LANGCHAIN_TRACING_V2"] = "false"
from ragas.llms  import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

In [84]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
ragas_embedding = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

C:\Users\Anurag\AppData\Local\Temp\ipykernel_33096\846061411.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
C:\Users\Anurag\AppData\Local\Temp\ipykernel_33096\846061411.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embedding = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [76]:
list_of_50_data_set=[]

In [80]:
def get_50_result(reference_input):
    # Pass qdrant_client to reuse the keep-alive connection pool
    res = rag_pipeline(reference_input["question"], qdrant_client=qdrant_client)
    list_of_50_data_set.append(res)


In [81]:
from tqdm import tqdm

list_of_50_data_set = []

for i in tqdm(range(50)):
    try:
        get_50_result(output_data[i].inputs)
    except Exception as e:
        print(f"Error on index {i}: {e}")

100%|██████████| 50/50 [04:52<00:00,  5.84s/it]


In [82]:
list_of_50_data_set

[{'original_output': RAGGenerationResponse(answer='The hidden cameras that include motion detection and push notifications to apps are:\n\n1. 4K Smoke Detector WiFi Hidden Cameras (ID: B09KV7HQK9) - Motion sensing and alert push notifications to your phone.\n2. Aipmoz Hidden Camera in Bluetooth Speaker (ID: B0CB1XQ9NR) - Motion activated with push alarm notifications.\n3. Mini Spy Camera WiFi Wireless Hidden Cameras (ID: B0C7HD4DY2) - Motion detection alert with push notifications to your phone.\n4. Gellisleft Hidden Camera Clock (ID: B0BW8SWK6D) - Motion detection with alarm notifications pushed to your mobile app.\n5. OKSIR Hidden Spy Camera USB Charger (ID: B0BVZB912Z) - Motion detection with push notifications sent to your cellphone.', references=[RAGUsedContext(id='B09KV7HQK9', description="4K Smoke Detector WiFi Hidden Cameras HD 1080P Small Camera Night Vision and Motion Detection Spy Cameras for Home Security Nanny CamsFull HD camera: 1080P HD resolution allows you to clearly s

In [64]:
async def ragas_faithfullness(run,example):
    sample=SingleTurnSample(
        user_input=run['Question'],
        response=run['answer'],
        retrieved_contexts=run['reterived_context']
    )
    scorer=Faithfulness(llm=ragas_llm)
    return await scorer.single_turn_ascore(sample)

In [ ]:
result=await ragas_faithfullness(res,"")

In [65]:
async def ragas_responcereleveancy(run,example):
    sample=SingleTurnSample(
        user_input=run['Question'],
        response=run['answer'],
        retrieved_contexts=run['reterived_context']
    )
    scorer=ResponseRelevancy(llm=ragas_llm,embeddings=ragas_embedding)
    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_responcereleveancy(res,"")

In [66]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision

C:\Users\Anurag\AppData\Local\Temp\ipykernel_33096\1969490736.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision


In [67]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision,IDBasedContextRecall

async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["reterived_context_ids"],
        reference_context_ids=example["refrence_context_ids"],
    )

    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)

C:\Users\Anurag\AppData\Local\Temp\ipykernel_33096\3985038912.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision,IDBasedContextRecall
C:\Users\Anurag\AppData\Local\Temp\ipykernel_33096\3985038912.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision,IDBasedContextRecall


In [85]:
import asyncio
import numpy as np
import pandas as pd
from tqdm import tqdm
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    IDBasedContextPrecision,
    IDBasedContextRecall,
)

# 1. Initialize the 4 Scorers
faithfulness_scorer = Faithfulness(llm=ragas_llm)
relevancy_scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embedding)
precision_scorer = IDBasedContextPrecision()
recall_scorer = IDBasedContextRecall()

# 2. Evaluation helper for a single sample
async def evaluate_sample(run, example_outputs):
    # Safely handle field spellings ('reference_context_ids' vs 'refrence_context_ids')
    ref_ids = (
        example_outputs.get("reference_context_ids")
        or example_outputs.get("refrence_context_ids")
        or []
    )

    sample = SingleTurnSample(
        user_input=run.get("Question") or run.get("question", ""),
        response=run.get("answer", ""),
        retrieved_contexts=run.get("reterived_context") or run.get("retrieved_context") or [],
        retrieved_context_ids=run.get("reterived_context_ids") or run.get("retrieved_context_ids") or [],
        reference_context_ids=ref_ids,
    )

    # Safe score wrapper to prevent a single failure from crashing the batch
    async def safe_score(scorer):
        try:
            score = await scorer.single_turn_ascore(sample)
            return float(score) if score is not None and not np.isnan(score) else np.nan
        except Exception as e:
            return np.nan

    f_score = await safe_score(faithfulness_scorer)
    r_score = await safe_score(relevancy_scorer)
    p_score = await safe_score(precision_scorer)
    rec_score = await safe_score(recall_scorer)

    return {
        "question": sample.user_input,
        "faithfulness": f_score,
        "response_relevancy": r_score,
        "context_precision": p_score,
        "context_recall": rec_score,
    }

# 3. Run evaluation across all 50 datasets
eval_results = []
num_samples = min(len(list_of_50_data_set), len(output_data))

print(f"Starting evaluation of {num_samples} datasets...")
for i in tqdm(range(num_samples), desc="Evaluating RAGAS Metrics"):
    run = list_of_50_data_set[i]
    example_outputs = output_data[i].outputs
    res = await evaluate_sample(run, example_outputs)
    eval_results.append(res)

# 4. Convert to DataFrame
df_eval = pd.DataFrame(eval_results)

# 5. Compute average scores & percentages
metrics_map = {
    "faithfulness": "Faithfulness",
    "response_relevancy": "Response Relevancy",
    "context_precision": "Context Precision (ID-Based)",
    "context_recall": "Context Recall (ID-Based)",
}

summary_rows = []
for col, display_name in metrics_map.items():
    mean_val = df_eval[col].dropna().mean()
    pct_val = mean_val * 100
    summary_rows.append({
        "Eval Type": display_name,
        "Average Score (0-1)": round(mean_val, 4),
        "Average Percentage": f"{pct_val:.2f}%",
        "Evaluated Samples": int(df_eval[col].dropna().count())
    })

# Overall average across all metrics
overall_mean = df_eval[list(metrics_map.keys())].mean().mean()
summary_rows.append({
    "Eval Type": "Overall Average",
    "Average Score (0-1)": round(overall_mean, 4),
    "Average Percentage": f"{overall_mean * 100:.2f}%",
    "Evaluated Samples": num_samples
})

summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 65)
print("              RAG EVALUATION SUMMARY (50 SAMPLES)")
print("=" * 65)
display(summary_df)

C:\Users\Anurag\AppData\Local\Temp\ipykernel_33096\3728111261.py:6: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\Anurag\AppData\Local\Temp\ipykernel_33096\3728111261.py:6: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import (
C:\Users\Anurag\AppData\Local\Temp\ipykernel_33096\3728111261.py:6: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import (
C:\Users\Anurag\AppData\Local

Starting evaluation of 50 datasets...


Evaluating RAGAS Metrics: 100%|██████████| 50/50 [18:42<00:00, 22.45s/it]


              RAG EVALUATION SUMMARY (50 SAMPLES)


,Eval Type,Average Score (0-1),Average Percentage,Evaluated Samples
0,Faithfulness,0.7274,72.74%,50
1,Response Relevancy,0.7474,74.74%,50
2,Context Precision (ID-Based),0.2070,20.70%,50
3,Context Recall (ID-Based),0.7823,78.23%,49
4,Overall Average,0.6160,61.60%,50
